# Structured Data

When you ask Claude for JSON (or code, or a list), it likes to be helpful and wrap the content in a markdown code block **plus** an explanatory sentence. Great for a chat UI; annoying when your app needs to copy/paste or `json.loads()` the raw data.

The course's technique: **assistant message prefilling + a stop sequence.**

1. The user message says what to generate.
2. A prefilled assistant message (` ```json `) makes Claude think it already opened a code block, so it continues with just the content.
3. A `stop_sequences=["```"]` ends generation the instant Claude tries to close the fence.

> ⚠️ **Model note.** Assistant prefilling returns a **400** on the flagship models (`claude-sonnet-5`, `claude-opus-4-8`, …) — Anthropic removed last-assistant-turn prefills. `stop_sequences` still works everywhere. To run the course technique we use **`claude-haiku-4-5-20251001`**, which still supports prefilling. The last cell shows the modern, prefill-free alternative for flagship models.

## Setup + a `chat()` with `stop_sequences`

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())   # find the repo-root .env from any folder depth

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"   # supports assistant prefilling


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

## The problem — default response

Ask for JSON normally and watch Claude wrap it in a ```` ```json ```` fence with an explanatory sentence after it.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

print(chat(messages))

## The fix — prefill + stop sequence

Prefill the assistant turn with ```` ```json ```` and stop on ```` ``` ````. The response comes back as just the JSON body (Claude never gets to write the explanation).

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)

## Parse it

There may be stray leading/trailing newlines — `.strip()` handles them, then `json.loads()` parses. If this returns a real dict, you have clean, usable structured data.

In [ ]:
import json

clean_json = json.loads(text.strip())
clean_json

## Bonus — the modern, prefill-free way (works on flagship models)

Since prefilling 400s on `claude-sonnet-5` / `claude-opus-4-8`, the current way to *guarantee* clean JSON is **structured outputs**: pass a JSON schema via `output_config`, and the API constrains the response to valid JSON matching it — no markdown fence, no explanation, no stop-sequence trick. This is what the *Structured data exercise* (next lesson) leans toward.

**Schema rule that just bit us:** every `object` in the schema must set `additionalProperties: false` — the API rejects `true`. So a truly free-form field (like EventBridge's `detail`) has to be given a concrete shape rather than "any keys allowed."

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "source": {"type": "array", "items": {"type": "string"}},
        "detail-type": {"type": "array", "items": {"type": "string"}},
        "detail": {
            "type": "object",
            "properties": {
                "state": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["state"],
            "additionalProperties": False,
        },
    },
    "required": ["source", "detail-type", "detail"],
    "additionalProperties": False,
}

response = client.messages.create(
    model="claude-sonnet-5",   # flagship model; no prefill needed
    max_tokens=1000,
    messages=[{"role": "user", "content": "Generate a very short EventBridge rule for EC2 instances entering the running state."}],
    output_config={"format": {"type": "json_schema", "schema": schema}},
)

# output_config guarantees the text block is valid JSON matching the schema
raw = next(b.text for b in response.content if b.type == "text")
json.loads(raw)